In [30]:
import pickle
import numpy as np
import random
import os
import gzip
import torch

def generate_examples(example):
    colorings,label,A,b,target = example['colorings'],example['labels'],example['A'],example['b'],example['target']
    perm = np.random.permutation(len(colorings))
    available = [colorings[i] for i in perm]
    available_labels = [label[i] for i in perm]
    datapoint = {'available':np.stack(available,axis=0),
                'available_labels':np.array(available_labels),
                'target':np.array(target),
                'graph_matrix':A,
                'b':b}
    return datapoint


indices = list(range(10)) + ["max"]
for index in indices:
    # Load the graph coloring data
    with open(f'../temp/solver_data/variations2/noise_level_{index}.pkl', 'rb') as f:
        data = pickle.load(f)

    examples = []
    for example in data:
        examples.append(generate_examples(example))
    random.shuffle(examples)
    root = f'/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/generate_column_noise_level_{index}'
    subfolder = 'sub'
    batch_size = 1000
    total_problems = 15000
    os.makedirs(f'{root}/{subfolder}/raw', exist_ok=True)
    ips = []
    pkg_idx = 0
    for ex in examples[:total_problems]:
        available = ex['available'].transpose()
        labels = ex['available_labels']
        target = ex['target']
        graph_matrix = ex['graph_matrix']
        b = ex['b']

        ips.append({'available':torch.from_numpy(available).to(torch.float), 
                    'available_labels':torch.from_numpy(labels).to(torch.float), 
                    'target':torch.from_numpy(target).to(torch.float).squeeze(0),
                    'graph_matrix':torch.from_numpy(graph_matrix).to(torch.float),
                    'b':torch.from_numpy(b).to(torch.float)}) 
        if len(ips) >= batch_size:
                with gzip.open(f'{root}/{subfolder}/raw/instance_{pkg_idx}.pkl.gz', "wb") as file:
                    pickle.dump(ips, file)
                    pkg_idx += 1
                ips = []


In [28]:
ips[0]['target'].shape

torch.Size([17])

In [21]:
ips[0]['available_labels'].shape

torch.Size([6])